<a href="https://colab.research.google.com/github/Iberasa3/BlackAndOchre/blob/main/Avispas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vespa velutina macro analysis notebook:

## This section is basic dataframe processing and analysis


In [18]:
import pandas as pd

# The '\t' separator tells pandas to use the tab key as a delimiter
df = pd.read_csv('avispas_avistamientos.csv', sep='\t')
print(f"Number of columns: {df.shape[1]}")
print(f"Column names: {df.columns.tolist()[:5]}...")

/tmp/ipykernel_303/1355496833.py:4: DtypeWarning: Columns (14,16,17,36,37,38,39,40,41,43,44,45,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('avispas_avistamientos.csv', sep='\t')


Number of columns: 50
Column names: ['gbifID', 'datasetKey', 'occurrenceID', 'kingdom', 'phylum']...


In [19]:
df['infraspecificEpithet'].unique() # Just cheking how many subespecies there are

array([nan, 'nigrithorax', 'variana', 'auraria', 'flavitarsus',
       'velutina', 'karnyi', 'celebensis', 'ardens', 'sumbana',
       'divergens', 'floresiana', 'timorensis'], dtype=object)

In [20]:
print(df['infraspecificEpithet'].value_counts(dropna=False)) #Checking instances of subespecies

infraspecificEpithet
NaN            306920
nigrithorax     70676
auraria           101
variana            46
flavitarsus        42
velutina           25
ardens             24
timorensis         16
karnyi             15
floresiana         13
celebensis         10
divergens           6
sumbana             5
Name: count, dtype: int64


In [21]:
europe = ['ES', 'FR', 'PT', 'BE', 'IT', 'DE', 'CH', 'LU', 'GB', 'IE', 'NL']
# The analysis is focused on Europe, so we are keeping only European countries.
# Most of the records are European anyway.
df = df[df['countryCode'].isin(europe)]

print(f"Number of records changed from {len(df)} to {len(df)}.")

Number of records changed from 372895 to 372895.


In [22]:
# Counting sightings per country and sorting from highest to lowest
country_counts = df['countryCode'].value_counts()

print("Sightings by Country in Europe")
print(country_counts)

Sightings by Country in Europe
countryCode
BE    201699
NL     72038
FR     54466
CH     20092
DE     17921
ES      3518
PT      2251
LU       642
IT       211
IE        37
GB        20
Name: count, dtype: int64


In [23]:
# Cleaning coordinates: removing zero values and nulls
df = df[(df['decimalLatitude'] != 0.0) & (df['decimalLongitude'] != 0.0)]
df = df.dropna(subset=['decimalLatitude', 'decimalLongitude'])

print(f"Original records: {len(df)}")
print(f"Records after removing nulls and null-islands: {len(df)}")

Original records: 372450
Records after removing nulls and null-islands: 372450


In [24]:
# Removing instances where uncertainty exceeds 1km.
max_threshold = 1000
df = df[(df['coordinateUncertaintyInMeters'] <= max_threshold) & (df['year'] >= 2005)]

In [25]:
df = df[df['basisOfRecord'] != 'PRESERVED_SPECIMEN']
print(df['basisOfRecord'].unique())

['HUMAN_OBSERVATION']


## Spatial Thinning and Data Independence

We are implementing **spatial thinning** to address spatial autocorrelation. Since we are using a **Random Forest** model, we must satisfy the assumption of independence between observations. Clustered points often reflect sightings from the same colony rather than independent distribution events.

Given that *Vespa velutina* typically forages within a **1 to 1.5 km radius** of its nest, sightings within this range are likely linked to the same location. By applying spatial thinning at this scale, we ensure that:
1. We avoid **oversampling** in highly monitored areas.
2. We treat the project as a **distribution model** (presence/absence) rather than an abundance model.
3. The spatial resolution remains consistent with the species' biological range.



In [26]:
# Checking for and removing duplicate records based on gbifID
num_duplicates = df.duplicated(subset='gbifID').sum()
print(f"Records with the same gbifID: {num_duplicates}")

# Keeping only the first occurrence of each unique ID
df.drop_duplicates(subset='gbifID', inplace=True)

Records with the same gbifID: 0


In [27]:
# 0.015 degrees of latitude is approximately 1.1 km.
# Each grid cell will be roughly 1.1 km x 1.1 km,
# aligning with the standard 1km resolution for bioclimatic models.
res = 0.015

# Divide by resolution, round to the nearest integer, and multiply back.
df['lat_grid'] = (df['decimalLatitude'] / res).round() * res
df['lon_grid'] = (df['decimalLongitude'] / res).round() * res

print(f"Original record count: {len(df)}")

# Dropping spatial duplicates within the same grid cell
df = df.drop_duplicates(subset=['lat_grid', 'lon_grid'], keep='first').copy()

print(f"Dataset after thinning (1km): {len(df)} records")

df[['decimalLatitude', 'lat_grid', 'decimalLongitude', 'lon_grid']].head()

Original record count: 231961
Dataset after thinning (1km): 29307 records


,decimalLatitude,lat_grid,decimalLongitude,lon_grid
2,51.11106,51.105,4.07784,4.080
3,51.21826,51.225,2.90049,2.895
4,50.86031,50.865,4.62720,4.620
5,51.03947,51.045,3.74158,3.735
6,50.95183,50.955,3.68754,3.690


We have reduced the dataset by nearly 90% by removing extremely close sightings. We are now left with 29,300 records to train our model.

In [28]:
# Counting how many records exist per country and sorting from highest to lowest.
# These represent the specific grid cells where sightings have occurred.
country_counts = df['countryCode'].value_counts()
total_count = df

print("--- Sightings by Country in Europe ---")
print(country_counts)

--- Sightings by Country in Europe ---
countryCode
BE    10519
FR     8792
DE     7482
ES     1247
PT      760
LU      184
NL      124
CH       94
IT       91
IE        9
GB        5
Name: count, dtype: int64


## Getting presences and their geographical data


This section handles the integration with Google Earth Engine (GEE) by authenticating and initializing the API. The local dataset is streamlined to include only essential spatial variables (latitude, longitude, and year) and is then converted into an Earth Engine FeatureCollection. This transformation shifts the data from a local Python environment to the cloud, enabling a interactive mapping of Vespa velutina occurrences across Europe

In [29]:
import sys
import geemap
import ee  # Import the Earth Engine library

# Install geojson library if not already installed
!pip install geojson

# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='vespa-489513')

# Filtering only the required columns for the spatial model
required_columns = ['decimalLatitude', 'decimalLongitude', 'year']
df = df[required_columns].copy()

# Converting the pandas DataFrame to an Earth Engine FeatureCollection
# These will be our spatial points for all subsequent analysis
geospatial_points = geemap.pandas_to_ee(
    df,
    latitude='decimalLatitude',
    longitude='decimalLongitude'
)

"""
UNCOMMENT THIS IF YOU WANT TO SEE THE MAP OF PRESENCES


# Visualizing the points on an interactive map
Map = geemap.Map()
Map.addLayer(geospatial_points, {'color': 'red'}, 'Vespa velutina (Optimized)')
Map.centerObject(geospatial_points, 6)
Map
"""

"\nUNCOMMENT THIS IF YOU WANT TO SEE THE MAP OF PRESENCES\n\n\n# Visualizing the points on an interactive map\nMap = geemap.Map()\nMap.addLayer(geospatial_points, {'color': 'red'}, 'Vespa velutina (Optimized)')\nMap.centerObject(geospatial_points, 6)\nMap\n"

--------------------------------------------

Given that the existing literature provides clear indicators of the environmental requirements for *Vespa velutina*, we will bypass **Principal Component Analysis (PCA)** in favor of a more interpretable approach. Instead, we will perform a **Multicollinearity Analysis** using the **Variance Inflation Factor (VIF)** to ensure our predictors are independent.
The candidate variables for the model include:

* **Bioclimatic (WorldClim):** Mean annual temperature (**bio01**), Max temperature of the warmest month (**bio05**), Min temperature of the coldest month (**bio06**), and Annual temperature range (**bio07**).
* **Precipitation (WorldClim):** Annual precipitation (**bio12**), Precipitation of the wettest month (**bio13**), Precipitation of the driest month (**bio14**), and Precipitation seasonality (**bio15**).
* **Topography (NASA SRTM):** Elevation and Slope.
* **Land Cover & Infrastructure:** ESA WorldCover (land type) and proximity to roads or urban areas (OpenStreetMap/GEE).

In [30]:
# PREDICTOR CONFIGURATION

# Define the Area of Interest (AOI) with a 50km buffer
aoi = geospatial_points.geometry().bounds().buffer(50000)

# Bioclimatic variables (WorldClim)
bioclim = ee.Image('WORLDCLIM/V1/BIO').clip(aoi)

# Topography variables (SRTM)
srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi)
elevation = srtm.select('elevation')
slope = ee.Terrain.slope(srtm).rename('slope')

# Land cover (ESA WorldCover)
land_cover = ee.ImageCollection("ESA/WorldCover/v100").first().clip(aoi).select(['Map'], ['landcover'])

# Infrastructure (Calculating distance to built-up areas/roads)
# Note: Using class 50 (Built-up) from WorldCover to estimate urban proximity
dist_roads = land_cover.eq(50).fastDistanceTransform().sqrt().multiply(1000).rename('dist_roads')

# Merging all bands into a single multi-dimensional image stack
predictors = bioclim.addBands(elevation).addBands(slope).addBands(land_cover).addBands(dist_roads).select([
    'bio01', 'bio05', 'bio06', 'bio07', 'bio12', 'bio13', 'bio14', 'bio15',
    'elevation', 'slope', 'landcover', 'dist_roads'
])

# SAMPLING
# Extracting environmental data for each presence point (Climate + Topography + Land Cover + Proximity)
sampled_presence_data = predictors.sampleRegions(
    collection=geospatial_points,
    properties=['year'],
    scale=1000,
    geometries=True
)

### MAP visualization of some useful environmental data (uncomment to see results)

In [31]:

"""
predictors = bioclim.addBands(elevation).addBands(slope).addBands(land_cover).addBands(dist_roads).select([
    'bio01', 'bio05', 'bio06', 'bio07', 'bio12', 'bio13', 'bio14', 'bio15', 'elevation', 'slope', 'landcover', 'dist_roads'
])

# Temperature Palette (Bio01, Bio06, etc.) - From cold blue to heat red
vis_temp = {'min': -50, 'max': 300, 'palette': ['blue', 'cyan', 'green', 'yellow', 'red']}

# Precipitation Palette (Bio12, Bio14) - From dry white to rainfall blue
vis_precip = {'min': 0, 'max': 2500, 'palette': ['white', 'lightblue', 'blue', 'darkblue']}

# Elevation Palette
vis_el = {'min': 0, 'max': 2500, 'palette': ['006600', '002200', 'fff700', 'ab7634', 'c7d6ec', 'ffffff']}

# Slope Palette - From gentle green to steep red
vis_slope = {'min': 0, 'max': 45, 'palette': ['white', 'orange', 'red']}

# Official ESA WorldCover Palette (Landcover)
vis_lc = {'bands': ['landcover'], 'min': 10, 'max': 100,
          'palette': ['006400', 'ffbb22', 'ffff4c', 'f096ff', 'fa0000', 'b4b4b4', 'f0f0f0', '0064ff', '00bb88', 'ffff4c']}

# Distance to Roads Palette - From red (close) to white (far)
vis_dist = {'min': 0, 'max': 10000, 'palette': ['red', 'orange', 'white']}

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Adding layers one by one (toggle them in the layer menu)
Map.addLayer(predictors.select('bio01'), vis_temp, 'Annual Mean Temperature (Bio01)', False)
Map.addLayer(predictors.select('bio06'), vis_temp, 'Min Temperature of Coldest Month (Bio06)', False)
Map.addLayer(predictors.select('bio12'), vis_precip, 'Annual Precipitation (Bio12)', False)
Map.addLayer(predictors.select('elevation'), vis_el, 'Elevation (SRTM)', False)
Map.addLayer(predictors.select('slope'), vis_slope, 'Slope')
Map.addLayer(predictors.select('landcover'), vis_lc, 'Land Cover (ESA WorldCover)')
Map.addLayer(predictors.select('dist_roads'), vis_dist, 'Proximity to Urban/Road Areas')

# Adding presence points as a visual reference
Map.addLayer(geospatial_points, {'color': 'magenta'}, 'Actual Nests (Points)')

# Center the map
Map.centerObject(geospatial_points, 7)
Map
"""



"\npredictors = bioclim.addBands(elevation).addBands(slope).addBands(land_cover).addBands(dist_roads).select([\n    'bio01', 'bio05', 'bio06', 'bio07', 'bio12', 'bio13', 'bio14', 'bio15', 'elevation', 'slope', 'landcover', 'dist_roads'\n])\n\n# Temperature Palette (Bio01, Bio06, etc.) - From cold blue to heat red\nvis_temp = {'min': -50, 'max': 300, 'palette': ['blue', 'cyan', 'green', 'yellow', 'red']}\n\n# Precipitation Palette (Bio12, Bio14) - From dry white to rainfall blue\nvis_precip = {'min': 0, 'max': 2500, 'palette': ['white', 'lightblue', 'blue', 'darkblue']}\n\n# Elevation Palette\nvis_el = {'min': 0, 'max': 2500, 'palette': ['006600', '002200', 'fff700', 'ab7634', 'c7d6ec', 'ffffff']}\n\n# Slope Palette - From gentle green to steep red\nvis_slope = {'min': 0, 'max': 45, 'palette': ['white', 'orange', 'red']}\n\n# Official ESA WorldCover Palette (Landcover)\nvis_lc = {'bands': ['landcover'], 'min': 10, 'max': 100,\n          'palette': ['006400', 'ffbb22', 'ffff4c', 'f096ff'

### VIF

In this section, we call the **VIF_variable_selector** script to perform a feature selection. This process identifies the variables that best represent our data from the set of all potentially useful predictors we previously considered. Once we know the golden set there is no need to execute this, but you can uncomment it to see the results!

In [32]:
"""
from VIF_variable_selector import run_vif_cleaner

print("Downloading data from GEE...")
df_total = geemap.ee_to_df(sampled_presence_data)

# Setting the threshold to 10.0, though 5.0 could also be considered.
threshold = 10.0

print("Starting redundancy purge...")
final_variables, vif_report = run_vif_cleaner(df_total, threshold)

print("\n--- GOLDEN SUBSET ---")
print(f"Surviving variables: {final_variables}")
print("\nFinal VIF values:")
print(vif_report)
"""

'\nfrom VIF_variable_selector import run_vif_cleaner\n\nprint("Downloading data from GEE...")\ndf_total = geemap.ee_to_df(sampled_presence_data)\n\n# Setting the threshold to 10.0, though 5.0 could also be considered.\nthreshold = 10.0\n\nprint("Starting redundancy purge...")\nfinal_variables, vif_report = run_vif_cleaner(df_total, threshold)\n\nprint("\n--- GOLDEN SUBSET ---")\nprint(f"Surviving variables: {final_variables}")\nprint("\nFinal VIF values:")\nprint(vif_report)\n'

This has been used to generate the graphs present in the documentation.
You can uncomment it too

In [33]:
"""
import matplotlib.pyplot as plt

data = {
    'Variable': ['bio05', 'bio06', 'bio13', 'bio14', 'elevation', 'bio01', 'bio15', 'slope', 'bio07', 'bio12', 'dist_roads', 'landcover'],
    'VIF': [500, 104.91, 24.71, 10.16, 2.86, 2.36, 2.34, 1.80, 1.61, 1.35, 1.31, 1.11],
    'Subset': ['Dropped', 'Dropped', 'Dropped', 'Dropped', 'Golden', 'Golden', 'Golden', 'Golden', 'Golden', 'Golden', 'Golden', 'Golden']
}

df = pd.DataFrame(data)
# Invert the order so the "Golden" subset appears at the top of the horizontal chart
df = df.iloc[::-1]

# Red for Dropped, Blue for Golden
colors = ['#e74c3c' if s == 'Dropped' else '#3498db' for s in df['Subset']]

plt.figure(figsize=(10, 8))

# Create horizontal bar chart
bars = plt.barh(df['Variable'], df['VIF'], color=colors, edgecolor='black', alpha=0.8)

# Apply log scale to the X-axis for better visibility of low VIF values
plt.xscale('log')

# Add VIF value labels to the end of each bar
for bar in bars:
    xval = bar.get_width()
    plt.text(xval * 1.1, bar.get_y() + bar.get_height()/2,
             f'{xval if xval < 400 else "inf"}',
             va='center', ha='left', fontsize=10, fontweight='bold')

# Add vertical line at the VIF=10 threshold
plt.axvline(x=10, color='gray', linestyle='--', alpha=0.6, label='Threshold (VIF=10)')

plt.title('VIF Analysis: Feature Selection for Vespa velutina Model', fontsize=14, pad=20)
plt.xlabel('VIF Value (Log Scale)')
plt.ylabel('Environmental Predictors')

plt.grid(axis='x', linestyle=':', alpha=0.5)

# Custom legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color='#3498db', lw=4, label='Golden Subset (Accepted)'),
                   Line2D([0], [0], color='#e74c3c', lw=4, label='Redundant (Dropped)')]
plt.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

"""

'\nimport matplotlib.pyplot as plt\n\ndata = {\n    \'Variable\': [\'bio05\', \'bio06\', \'bio13\', \'bio14\', \'elevation\', \'bio01\', \'bio15\', \'slope\', \'bio07\', \'bio12\', \'dist_roads\', \'landcover\'],\n    \'VIF\': [500, 104.91, 24.71, 10.16, 2.86, 2.36, 2.34, 1.80, 1.61, 1.35, 1.31, 1.11],\n    \'Subset\': [\'Dropped\', \'Dropped\', \'Dropped\', \'Dropped\', \'Golden\', \'Golden\', \'Golden\', \'Golden\', \'Golden\', \'Golden\', \'Golden\', \'Golden\']\n}\n\ndf = pd.DataFrame(data)\n# Invert the order so the "Golden" subset appears at the top of the horizontal chart\ndf = df.iloc[::-1]\n\n# Red for Dropped, Blue for Golden\ncolors = [\'#e74c3c\' if s == \'Dropped\' else \'#3498db\' for s in df[\'Subset\']]\n\nplt.figure(figsize=(10, 8)) \n\n# Create horizontal bar chart\nbars = plt.barh(df[\'Variable\'], df[\'VIF\'], color=colors, edgecolor=\'black\', alpha=0.8)\n\n# Apply log scale to the X-axis for better visibility of low VIF values\nplt.xscale(\'log\')\n\n# Add VIF val

### Feature Filtering: Final Presence Selection

In this section, we filter the presence records to retain only the specific environmental variables of interest for our model based on their VIF

In [34]:
# Variables that maintained a VIF < 10 (FINAL PRESENCE COLLECTION)
golden_variables = [
    'elevation', 'bio01', 'bio15', 'slope',
    'bio07', 'bio12', 'dist_roads', 'landcover'
] # dist_roads sometimes causes bugs; we can re-insert it later if needed

# Filter the multi-band image to keep only the selected predictors
final_predictors = predictors.select(golden_variables)

# Sampling the final environmental data at presence locations
final_presence_data = final_predictors.sampleRegions(
    collection=geospatial_points,
    properties=['year'],
    scale=1000,
    geometries=True
)

# Labeling these records as class 1 (Presence)
final_presence_data = final_presence_data.map(lambda f: f.set('class', 1))

----------------------------------------------------------------

# Generating pseudoabsences

## SM1

Estrategia para generar pseudoausencias 1: SM1 random
Generamos puntos random en nuestra zona de interés que no coincidan con los puntos de presencia. Una vez generados los visualizamos y guardamos en un dataset para poder compararlos en un futuro y no tener que ejecutar esta sección cada vez.

In [42]:

# SM1 PSEUDO-ABSENCE GENERATION (CRITICAL STEP: VERIFYING THE PSEUDO-ABSENCE COLLECTION)

# Define the study area with a 2,000 km buffer around the presence points
aoi = geospatial_points.geometry().bounds().buffer(2000000)

# Create an image from presence points
presence_img = ee.Image().paint(geospatial_points, 1000)

# Create a land mask to avoid generating points in the ocean (Class 80 in ESA WorldCover is water)
land_cover = ee.ImageCollection("ESA/WorldCover/v100").first()
land_mask = land_cover.neq(80)

# Create an exclusion mask to ensure pseudo-absences don't fall on presence locations
# We use a 1km buffer to avoid pixel overlap
exclusion_mask = presence_img.fastDistanceTransform().sqrt().gt(1)

# Sampling area: Allowed regions for absences (excluding presences and water)
sampling_area = ee.Image(1).clip(aoi).updateMask(exclusion_mask).updateMask(land_mask)

# Generate pseudo-absences aiming for a 1:1 ratio with presence points.
# The random points are generated within the allowed sampling area.
pseudo_absences_raw = final_predictors.updateMask(sampling_area).sample(
    region=aoi,
    scale=1000,
    numPixels=1000000, # This is truncated by the limit below
    seed=67,
    geometries=True
)

# Labeling and merging the dataset
# Class 1 = Presence, Class 0 = Pseudo-absence

# 1. Label absences as Class 0
# These already contain the 'golden_variables' from the sampling step above
absences = pseudo_absences_raw.limit(28887).map(lambda f: f.set('class', 0))
final_presence_data = final_presence_data.map(lambda f: f.set('class', 1))
total_dataset_SM1 = final_presence_data.merge(absences)


print(f"SM1 Dataset ready. ABSENCES: {absences.size().getInfo()} (Target: 29,306)")
print(f"SM1 Dataset ready. PRESENCES: {final_presence_data.size().getInfo()} (Target: 29,306)")
# print(f"SM1 Total points: {total_dataset_SM1.size().getInfo()} (Target: ~58,512)")


SM1 Dataset ready. ABSENCES: 28887 (Target: ~29,306)
SM1 Dataset ready. PRESENCES: 28887 (Target: ~29,306)


In [43]:

# SM1 VISUALIZATION SECTION
Map = geemap.Map()
Map.add_basemap('HYBRID')

# Styling for presence and absence points
vespa_style = {'color': 'red', 'pointSize': 4, 'pointShape': 'circle', 'width': 1}
absences_style = {'color': '00ffff', 'pointSize': 4, 'pointShape': 'square', 'width': 1} # Cyan squares

# Visualization parameters for the sampling mask to see where points can be placed
vis_mask = {
    'min': 0,
    'max': 1,
    'palette': ['000000', 'yellow'],
    'opacity': 0.35
}

# Add layers to the map
Map.addLayer(sampling_area, vis_mask, 'Debug: Sampling Area (Allowed Zones)')
Map.addLayer(absences.style(**absences_style), {}, 'Pseudo-absences (Class 0)')
Map.addLayer(geospatial_points.style(**vespa_style), {}, 'Vespa Presences (Class 1)')

Map.centerObject(geospatial_points, 6)
Map


Map(center=[48.71174353299487, 3.7533902755953057], controls=(WidgetControl(options=['position', 'transparent_…

In [50]:
"""
import geemap

# Define a color palette: from white/yellow (low probability) to dark red (high probability)
vis_params = {
    'min': 0,
    'max': 1,
    'palette': ['#ffffff', '#fdfd96', '#ffb347', '#ff6961', '#8b0000']
}

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Setting the classifier to PROBABILITY mode
# This generates a continuous Habitat Suitability Index (HSI) from 0 to 1
prob_classifier = classifier.setOutputMode('PROBABILITY')

# Classify the multi-band image using only the selected "Golden" predictors
suitability_map = final_predictors.select(golden_variables).classify(prob_classifier)

"""
"""
# Displaying the final suitability model
Map.addLayer(suitability_map, vis_params, 'Habitat Suitability Index (Probability)')

# Overlaying observed presence points to visually validate the model's accuracy
Map.addLayer(final_presence_data, {'color': 'blue'}, 'Observed Presences')

# Center the map on the study area
Map.centerObject(geospatial_points, 5)
Map
"""


NameError: name 'classifier' is not defined

Exportamos las pseudoausencias generadas con SM1 para no tener que hacer el proceso de limpieza cada vez y luego poder comparar los modelos utilizando diferentes técnicas.
Esto lo mantengo comentado por si acaso tengo que volver a hacerlo.



In [ ]:
"""
# EXTRACCIÓN Y GUARDADO DE COORDENADAS (APLICABLE PARA TODAS CAMBIANDO UN PAR DE COSILLAS)

def extraer_posicion(feature):
    coords = feature.geometry().coordinates()
    return feature.set({
        'latitude': coords.get(1),
        'longitude': coords.get(0)
    })


ausencias_finales = ausencias.map(extraer_posicion).select(['latitude', 'longitude', 'class'])
nombre_csv = 'SM1_absences_COORDENADASDef.csv'
geemap.ee_to_csv(ausencias_finales, filename=nombre_csv)
"""

## SM2

Estrategia para generar pseudoausencias 2: SM2 límite geográfico
Generamos puntos random en nuestra zona de interés que no coincidan con los puntos de presencia, pero SOLO en un área suficientemente cercana a nuestros puntos para obligar al modelo a aprender de zonas que estén relativamente cerca de nuestra área. No tiene tanto impacto porque se solapa bastante con aoi. Una vez generados los visualizamos y guardamos en un dataset para poder compararlos en un futuro y no tener que ejecutar esta sección cada vez.

In [67]:
# SM2 ABSENCE GENERATION (SPATIALLY CONSTRAINED)

# Define the AOI as a "cloud" of 350km buffers around the nests
# We remove .bounds() to use the actual area of influence instead of a bounding box
aoi_sm2 = geospatial_points.geometry().buffer(350000)

# Create an image from presence points
presence_img = ee.Image().paint(geospatial_points, 1)

# Create a land mask to avoid water bodies (Class 80 in ESA WorldCover)
land_cover = ee.ImageCollection("ESA/WorldCover/v100").first()
land_mask = land_cover.neq(80)

# Exclusion mask: Ensures pseudo-absences are not placed exactly on presence pixels
exclusion_mask = presence_img.fastDistanceTransform().sqrt().gt(1)

# Sampling area: Allowed regions for SM2 absences (constrained by the 350km buffer)
sampling_area_sm2 = ee.Image(1).clip(aoi_sm2).updateMask(exclusion_mask).updateMask(land_mask)

# SM2 GENERATION (CORREGIDO)

# 1. Muestrear las pseudo-ausencias DESDE los predictores
# Así cada punto ya trae las variables de oro ('bio01', 'elevation', etc.)
pseudo_absences_sm2_raw = final_predictors.updateMask(sampling_area_sm2).sample(
    region=aoi_sm2,
    scale=1000,
    numPixels=100000,
    seed=67,
    geometries=True
)

# 2. Truncar y etiquetar como Clase 0
absences_sm2 = pseudo_absences_sm2_raw.limit(29306).map(lambda f: f.set('class', 0))

# 3. ¡CUIDADO AQUÍ! Las presencias también deben tener los datos climáticos
# No uses 'geospatial_points' directamente, usa el objeto que ya muestreaste antes
# (Supongamos que se llama final_presence_data y ya tiene las variables_oro)
presencias_sm2 = final_presence_data.map(lambda f: f.set('class', 1))

# 4. Dataset final SM2
total_dataset_sm2 = presencias_sm2.merge(absences_sm2)

print(f"SM2 Listo. Cada punto tiene ahora las variables climáticas necesarias.")

SM2 Listo. Cada punto tiene ahora las variables climáticas necesarias.


In [58]:
""" APARTADO DE VISUALIZACIÓN SM2
# SM2 VISUALIZATION SECTION

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Defined styles for Vespa (Presences) and Absences
vespa_style = {'color': 'red', 'pointSize': 4, 'pointShape': 'circle', 'width': 1}
absence_style = {'color': '00ffff', 'pointSize': 4, 'pointShape': 'square', 'width': 1}

# Search Mask Configuration (350km "Cloud")
# unmask(0) ensures the 1km exclusion gaps appear black
# clip(aoi_sm2) ensures we see the actual cloud shape rather than a bounding box
vis_mask = {
    'min': 0,
    'max': 1,
    'palette': ['000000', 'yellow'], # Black for restricted, Yellow for allowed
    'opacity': 0.45 # Balanced for visibility and map clarity
}

# Representing the permitted sampling zone for SM2
Map.addLayer(sampling_area_sm2.unmask(0).clip(aoi_sm2), vis_mask, 'SM2 Sampling Region (350km Cloud)')

# Adding the points (styled for better performance and clarity)
Map.addLayer(absences_sm2.style(**absence_style), {}, 'SM2 Pseudo-absences (Class 0)')
Map.addLayer(presences.style(**vespa_style), {}, 'Vespa Presences (Class 1)')

# Center the map on the invasion front
Map.centerObject(geospatial_points, 6)
Map
"""

Map(center=[48.71174353299487, 3.7533902755953057], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
"""
# EXTRACCIÓN Y GUARDADO DE COORDENADAS (APLICABLE PARA TODAS CAMBIANDO UN PAR DE COSILLAS)

def extraer_posicion(feature):
    coords = feature.geometry().coordinates()
    return feature.set({
        'latitude': coords.get(1),
        'longitude': coords.get(0)
    })


ausencias_finales = ausencias_sm2.map(extraer_posicion).select(['latitude', 'longitude', 'class'])
nombre_csv = 'SM2_absences_COORDENADASDef.csv'
geemap.ee_to_csv(ausencias_finales, filename=nombre_csv)"""

## SM3: Cambiamos de clima para forzar a que el modelo se ajuste

# EN ESTA SECCIÓN DE AQUÍ ABAJO VAMOS A ENTRENAR EL RANDOM FOREST CON LOS MODELOS DE PRESENCIA Y CON LAS PSEUDOAUSENCIAS DEL SM1



In [51]:
import geemap

classifier = ee.Classifier.smileRandomForest(100).train(
    features = total_dataset_SM1,
    classProperty = 'class',
    inputProperties = golden_variables
)

# Define a color palette: from white/yellow (low probability) to dark red (high probability)
vis_params = {
    'min': 0,
    'max': 1,
    'palette': ['#ffffff', '#fdfd96', '#ffb347', '#ff6961', '#8b0000']
}

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Adding the Habitat Suitability Map
# Note: To see continuous probabilities (0 to 1) instead of binary classification,
# ensure the classifier is set to 'PROBABILITY' mode.
prob_classifier = classifier.setOutputMode('PROBABILITY')

# Classify the multi-band image using the final predictors
suitability_map = final_predictors.select(golden_variables).classify(prob_classifier)

"""
# Displaying the results
Map.addLayer(suitability_map, vis_params, 'Habitat Suitability Index (Probability)')
Map.addLayer(final_presence_data, {'color': 'blue'}, 'Observed Presences')

Map.centerObject(geospatial_points, 5)
Map
"""

"\n# Displaying the results\nMap.addLayer(suitability_map, vis_params, 'Habitat Suitability Index (Probability)')\nMap.addLayer(final_presence_data, {'color': 'blue'}, 'Observed Presences')\n\nMap.centerObject(geospatial_points, 5)\nMap\n"

In [52]:
bioclim_global = ee.Image('WORLDCLIM/V1/BIO')
srtm_global = ee.Image('USGS/SRTMGL1_003')
landcover_global = ee.ImageCollection("ESA/WorldCover/v100").first()

elev_global = srtm_global.select('elevation')
slope_global = ee.Terrain.slope(srtm_global).rename('slope')
# Calculamos la distancia a zonas urbanas (valor 50) en todo el mapa
dist_roads_global = landcover_global.eq(50).fastDistanceTransform().sqrt().multiply(1000).rename('dist_roads')

terreno_global = landcover_global.select(['Map'], ['landcover'])
predictores_global = bioclim_global.addBands(elev_global) \
                                   .addBands(slope_global) \
                                   .addBands(terreno_global) \
                                   .addBands(dist_roads_global)


# Usamos .classify() sobre la imagen global.
# El 'classifier' ya tiene el conocimiento guardado.
mapa_fitness_final = predictores_global.select(golden_variables).classify(classifier.setOutputMode('PROBABILITY'))

Map = geemap.Map()
Map.add_basemap('HYBRID')
Map.addLayer(mapa_fitness_final, {'min': 0, 'max': 1, 'palette': ['white', 'yellow', 'red']}, 'Fitness Vespa Global')
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

## SM2 PRUEBAS

In [68]:
import geemap

# Train the Random Forest classifier for the SM2 model
# Using 100 trees and the spatially constrained dataset (total_dataset_sm2)
classifier_sm2 = ee.Classifier.smileRandomForest(100).train(
    features = total_dataset_sm2,
    classProperty = 'class',
    inputProperties = golden_variables
)

# Define a color palette: from white/yellow (low probability) to dark red (high probability)
vis_params = {
    'min': 0,
    'max': 1,
    'palette': ['#ffffff', '#fdfd96', '#ffb347', '#ff6961', '#8b0000']
}

Map = geemap.Map()
Map.add_basemap('HYBRID')

# Switching the SM2 classifier to PROBABILITY mode
# This creates a continuous Habitat Suitability Index (HSI)
prob_classifier_sm2 = classifier_sm2.setOutputMode('PROBABILITY')

# Classify the multi-band predictor image using the SM2 model
suitability_map_sm2 = final_predictors.select(golden_variables).classify(prob_classifier_sm2)

"""
# Visualizing the SM2 results
Map.addLayer(suitability_map_sm2, vis_params, 'Habitat Suitability Index (SM2 - Constrained)')

# Overlaying observed presence points for visual validation
Map.addLayer(final_presence_data, {'color': 'blue'}, 'Observed Presences')

# Center the map on the study area
Map.centerObject(geospatial_points, 5)
Map
"""


"\n# Visualizing the SM2 results\nMap.addLayer(suitability_map_sm2, vis_params, 'Habitat Suitability Index (SM2 - Constrained)')\n\n# Overlaying observed presence points for visual validation\nMap.addLayer(final_presence_data, {'color': 'blue'}, 'Observed Presences')\n\n# Center the map on the study area\nMap.centerObject(geospatial_points, 5)\nMap\n"

In [70]:
# 1. Preparar predictores globales con nombres explícitos
bioclim_global = ee.Image('WORLDCLIM/V1/BIO')
srtm_global = ee.Image('USGS/SRTMGL1_003').select(['elevation']) # Forzamos nombre
land_cover_img = ee.ImageCollection("ESA/WorldCover/v100").first()

# 2. Generar capas derivadas
slope_global = ee.Terrain.slope(srtm_global).rename('slope')
# Importante: el landcover de ESA se llama 'Map', lo renombramos a 'landcover'
landcover_global = land_cover_img.select(['Map'], ['landcover'])
dist_roads_global = landcover_global.eq(50).fastDistanceTransform().sqrt().multiply(1000).rename('dist_roads')

# 3. Crear el stack final asegurando que los nombres coinciden con 'golden_variables'
global_predictors = bioclim_global \
    .addBands(srtm_global) \
    .addBands(slope_global) \
    .addBands(landcover_global) \
    .addBands(dist_roads_global)

# 4. Clasificación Global (SM2)
# Usamos el clasificador SM2 que ya "sabe" distinguir en distancias cortas
mapa_fitness_final_sm2 = global_predictors.select(golden_variables) \
    .classify(classifier_sm2.setOutputMode('PROBABILITY'))

# 5. Visualización
Map = geemap.Map()
Map.add_basemap('HYBRID')
Map.addLayer(mapa_fitness_final_sm2, {'min': 0, 'max': 1, 'palette': ['white', 'yellow', 'red']}, 'Fitness Vespa SM2 (Global)')
Map.centerObject(geospatial_points, 5)
Map

Map(center=[48.71174353299487, 3.7533902755953057], controls=(WidgetControl(options=['position', 'transparent_…

## REVISIONES

Cosas a arreglar o revisar:

5- ENTRENAR EL MODELO, LUEGO HACER VALIDACIÓN CRUZADA (SE VERÁ COMO SE HACE) PARA VER CÓMO DE BUENO ES.

6- REVISAR, DOCUMENTAR, README DE GITHUB


Enlaces útiles:

https://ramirodcrego.com/teaching/gee/

https://developers.google.com/earth-engine/tutorials/community/species-distribution-modeling?hl=es-419

https://www.tandfonline.com/doi/full/10.1080/10095020.2025.2546507#abstract

https://www.researchgate.net/publication/284246225_A_multi-scale_approach_to_identify_invasion_drivers_and_invaders'_future_dynamics

https://onlinelibrary.wiley.com/doi/full/10.1002/ece3.70029

https://besjournals.onlinelibrary.wiley.com/doi/10.1111/j.2041-210X.2011.00172.x

https://biogeography-usc.org/positive/Prediction.html

https://www.sciencedirect.com/science/article/abs/pii/S0006320711001315

https://revistaecosistemas.net/index.php/ecosistemas/article/view/2987/1962

https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0071218 (PAPER SM4)

https://gidahatari.com/ih-es/bioclim-un-sistema-de-analisis-y-prediccion-de-bioclimas (BIOCLIM)

https://www.researchgate.net/publication/226562656_DOMAIN_a_flexible_modelling_procedure_for_mapping_potential_distributions_of_plants_and_animals (DOMAIN)

https://www.sciencedirect.com/science/article/pii/S0304380011000780 (PAPER 1 JUSTIFICACIÓN BUFFER)

https://besjournals.onlinelibrary.wiley.com/doi/10.1111/j.2041-210X.2010.00036.x (PAPER 2 JUSTIFICACIÓN BUFFER)

https://besjournals.onlinelibrary.wiley.com/doi/full/10.1111/1365-2664.12724 (PAPER 3 JUSTIFICACIÓN BUFFER)

https://www.plant-animal.es/pdfs/Herrera.2024.Ecosistemas.pdf (paper justificación presencias)